# MLP Experience Replay
Train an MLPClassifier with year-wise incremental scaling and experience replay.

In [1]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [2]:
ds_path = Path('.') / 'training_data_with_features_plus_monthly_indices.zarr'
print(f'Loading data from {ds_path}...')
ds = xr.open_dataset(ds_path, engine='zarr')
print('Data loaded')

split_path = Path('.') / 'data_split.npz'
print(f'Loading split from {split_path}...')
split_data = np.load(split_path)
train_pixel_indices = split_data['train_pixel_indices']
val_pixel_indices = split_data['val_pixel_indices']
test_pixel_indices = split_data['test_pixel_indices']
print('Split loaded')

print('Dataset info:')
print(f'  Total pixels: {len(ds.pixel)}')
print(f'  Total years: {len(ds.year)}')
print(f'  Train pixels: {len(train_pixel_indices)}')
print(f'  Val pixels: {len(val_pixel_indices)}')
print(f'  Test pixels: {len(test_pixel_indices)}')

Loading data from training_data_with_features_plus_monthly_indices.zarr...


Data loaded
Loading split from data_split.npz...
Split loaded
Dataset info:
  Total pixels: 8155205
  Total years: 7
  Train pixels: 5597776
  Val pixels: 1273437
  Test pixels: 1283992


## 2. Feature Engineering

In [3]:
def prepare_raw_features_for_year(ds, pixel_indices, year_idx, s2_mean_per_pixel=None, dtype=np.float32):
    """Extract and clean raw (unscaled) features for one year. Year 0 is skipped by design."""
    if year_idx == 0:
        return np.empty((0, 0), dtype=dtype), np.empty((0,), dtype=np.int64)

    if s2_mean_per_pixel is None:
        s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
        s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    ds_subset = ds.isel(pixel=pixel_indices, year=year_idx)
    s2_features = ds_subset['s2_bands'].values
    if np.isnan(s2_features).any():
        s2_features = np.where(np.isnan(s2_features), s2_mean_per_pixel, s2_features)

    dem_features = ds_subset['dem'].values.reshape(-1, 1)
    ndvi_features = ds_subset['ndvi'].values.reshape(-1, 1)
    ndwi_features = ds_subset['ndwi'].values.reshape(-1, 1)

    ds_prev = ds.isel(pixel=pixel_indices, year=year_idx - 1)
    ndvi_last_year = np.where(np.isnan(ds_prev['ndvi'].values.reshape(-1, 1)), 0, ds_prev['ndvi'].values.reshape(-1, 1))
    ndwi_last_year = np.where(np.isnan(ds_prev['ndwi'].values.reshape(-1, 1)), 0, ds_prev['ndwi'].values.reshape(-1, 1))

    required_last_year_bands = ['B04', 'B03', 'B06']
    band_to_idx = {band: i for i, band in enumerate(ds['s2_band'].values)}
    missing_bands = [band for band in required_last_year_bands if band not in band_to_idx]
    if missing_bands:
        raise ValueError(f"Missing required S2 bands for last-year features: {missing_bands}")

    last_year_s2_features = []
    for band in required_last_year_bands:
        band_values = ds_prev['s2_bands'].sel(s2_band=band).values
        band_values = np.where(np.isnan(band_values), 0, band_values)
        last_year_s2_features.append(band_values.reshape(-1, 1))

    # Keep feature order aligned with MLP training notebook.
    features_list = [
        s2_features,
        dem_features,
        ndvi_features,
        ndwi_features,
        ndvi_last_year,
        ndwi_last_year,
        *last_year_s2_features,
    ]

    if 'nbr' in ds.data_vars:
        features_list.append(ds_subset['nbr'].values.reshape(-1, 1))

    if year_idx > 0 and 'ndvi_delta' in ds.data_vars:
        delta_year_idx = year_idx - 1
        ds_delta = ds.isel(pixel=pixel_indices, year=delta_year_idx)
        features_list.append(ds_delta['ndvi_delta'].values.reshape(-1, 1))
        features_list.append(ds_delta['ndwi_delta'].values.reshape(-1, 1))
        if 'nbr_delta' in ds.data_vars:
            features_list.append(ds_delta['nbr_delta'].values.reshape(-1, 1))

    for var_name in ['years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']:
        if var_name in ds.data_vars:
            features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    yearly_index_feature_names = [
        'ndvi_cv_year',
        'ndvi_max_m2m_drop_year',
        'ndvi_max_year',
        'ndvi_min_year',
        'ndvi_std_year',
        'ndwi_cv_year',
        'ndwi_max_m2m_drop_year',
        'ndwi_max_year',
        'ndwi_min_year',
        'ndwi_std_year',
    ]
    for feature_name in yearly_index_feature_names:
        if feature_name in ds.data_vars:
            features_list.append(ds_subset[feature_name].values.reshape(-1, 1))

    X = np.concatenate(features_list, axis=1)
    y = ds_subset['disturbances'].values

    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]

    nan_mask = ~np.isnan(X).any(axis=1)
    X_clean = X[nan_mask]
    y_clean = y[nan_mask]

    if len(X_clean) == 0:
        feature_dim = X.shape[1] if X.ndim == 2 and X.shape[0] > 0 else 0
        return np.empty((0, feature_dim), dtype=dtype), np.empty((0,), dtype=np.int64)

    return X_clean.astype(dtype, copy=False), y_clean.astype(np.int64, copy=False)


def prepare_features_for_year(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto', s2_mean_per_pixel=None, dtype=np.float32):
    """Extract, clean, and optionally scale features for one year. Year 0 is skipped by design."""
    valid_scaler_modes = {'auto', 'fit', 'partial_fit', 'transform', 'none'}
    if scaler_mode not in valid_scaler_modes:
        raise ValueError(f"Invalid scaler_mode '{scaler_mode}'. Valid options: {sorted(valid_scaler_modes)}")

    X_clean, y_clean = prepare_raw_features_for_year(
        ds,
        pixel_indices,
        year_idx,
        s2_mean_per_pixel=s2_mean_per_pixel,
        dtype=dtype,
    )

    if len(X_clean) == 0:
        return X_clean, y_clean, scaler

    if scaler_mode == 'none':
        return X_clean, y_clean, scaler

    if scaler is None:
        scaler = StandardScaler()

    if scaler_mode == 'auto':
        if hasattr(scaler, 'mean_'):
            X_clean = scaler.transform(X_clean)
        else:
            X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'fit':
        X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'partial_fit':
        scaler.partial_fit(X_clean)
        X_clean = scaler.transform(X_clean)
    elif scaler_mode == 'transform':
        if not hasattr(scaler, 'mean_'):
            raise ValueError("Scaler must be fitted before using scaler_mode='transform'.")
        X_clean = scaler.transform(X_clean)

    return X_clean, y_clean, scaler

## 3. Initialize Class Weights and MLP

In [4]:
print('Precomputing raw yearly features for train/validation splits...')
n_years = len(ds.year)

def precompute_yearly_raw_cache(ds, pixel_indices, n_years, split_name):
    s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    cache = {}
    empty_years = 0
    for year_idx in tqdm(range(1, n_years), desc=f'Precompute {split_name}'):
        X_raw, y_raw = prepare_raw_features_for_year(
            ds,
            pixel_indices,
            year_idx,
            s2_mean_per_pixel=s2_mean_per_pixel,
            dtype=np.float32,
        )
        cache[year_idx] = (X_raw, y_raw)
        if len(y_raw) == 0:
            empty_years += 1

    print(
        f"{split_name}: cached {len(cache)} years, empty years={empty_years}, "
        f"sample feature dim={next((x.shape[1] for x, y in cache.values() if len(y) > 0), 0)}"
    )
    return cache

train_feature_cache = precompute_yearly_raw_cache(ds, train_pixel_indices, n_years, 'train')
val_feature_cache = precompute_yearly_raw_cache(ds, val_pixel_indices, n_years, 'validation')

print('Computing class weights from cached training labels...')
train_label_batches = [y_batch for _, y_batch in train_feature_cache.values() if len(y_batch) > 0]
if not train_label_batches:
    raise ValueError('No valid training labels after filtering.')

all_train_labels = np.concatenate(train_label_batches).astype(int)
all_train_labels = all_train_labels[np.isin(all_train_labels, [0, 1])]
if len(all_train_labels) == 0:
    raise ValueError('No valid training labels after filtering.')

classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print('Class weights:')
print(f"  Class 0: {class_weight_dict[0]:.4f}")
print(f"  Class 1: {class_weight_dict[1]:.4f}")

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

Precomputing raw yearly features for train/validation splits...


Precompute train:   0%|          | 0/6 [00:00<?, ?it/s]

train: cached 6 years, empty years=0, sample feature dim=32


Precompute validation:   0%|          | 0/6 [00:00<?, ?it/s]

validation: cached 6 years, empty years=0, sample feature dim=32
Computing class weights from cached training labels...


Class weights:
  Class 0: 0.5105
  Class 1: 24.2384
MLP initialized


## 4. Replay-Enabled Online Training

### Replay Sampling Strategy (Uncertainty + Random)
For each training year, replay samples are split into two halves:
- **50% uncertainty-guided replay**
- **50% uniform random replay**

Uncertainty-guided half is sampled as follows:
1. Identify all previous years with available replay samples.
2. Split the uncertain replay target as evenly as possible across these previous years.
3. For each previous year, run current-model probability predictions on all that year's replay candidates.
4. Compute the **optimal F1 threshold** ($t^*$) for that year from probabilities and labels (threshold grid search).
5. Sample from that year's candidates using Gaussian weights centered on the year-specific threshold:

$$w_i = \exp\left(-\frac{(p_i - t^*)^2}{2\sigma^2}\right)$$

where $p_i$ is sample probability, and $t^*$ is the optimal F1 threshold.

To prevent probability underflow (which leads to sampling crashes when model predictions polarize during intermediate epochs), the standard deviation $\sigma$ is **adaptively adjusted** for each sampling step:

$$\sigma = \max(\sigma_{\text{base}}, d_{(K)} / 4.8)$$

where:
- $\sigma_{\text{base}}$ is the base configurable value (`UNCERTAINTY_GAUSSIAN_STD`, default 0.03),
- $d_{(K)}$ is the $K$-th smallest distance between prediction probabilities and the F1 threshold in the current pool,
- $K$ is the target sample quota for the year.

This adaptive logic guarantees that at least $K$ candidates retain non-trivial weights (specifically $\ge 10^{-5}$), ensuring robust sampling without underflow crashes.

The random half is sampled uniformly from remaining replay candidates to avoid overlap with uncertainty-selected samples.

In [5]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.2, 0.3, 0.4, 0.5]
REPLAY_ENABLED = True
REPLAY_RANDOM_STATE = 42
UNCERTAIN_REPLAY_FRACTION = 0.5
UNCERTAINTY_GAUSSIAN_STD = 0.03
UNCERTAINTY_THRESHOLD_GRID = np.linspace(0.0, 1.0, 201)
EPSILON = 1e-12

CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005

CHECKPOINT_DIR = Path('.') / 'training_checkpoints_mlp_experience_replay_uncertainity_prioritization'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories_uncertainity_prioritization.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status_uncertainity_prioritization.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_replay_training_log_uncertainity_prioritization.txt'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_uncertainity_prioritization_all_ratios.csv'

CHECKPOINT_DIR.mkdir(exist_ok=True)

if 'train_feature_cache' not in globals() or 'val_feature_cache' not in globals():
    raise ValueError('Raw feature caches not found. Run Cell 8 first to precompute yearly features.')

def create_empty_training_history():
    return {
        'year': [],
        'train_accuracy': [],
        'train_precision': [],
        'train_recall': [],
        'train_f1': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'val_roc_auc': [],
        'val_pr_auc': [],
        'replay_pool_size': [],
        'replay_target_size': [],
        'replay_used_size': [],
        'uncertain_target_size': [],
        'uncertain_used_size': [],
        'random_target_size': [],
        'random_used_size': [],
        'uncertain_threshold_mean': [],
    }

def load_all_training_histories(path):
    if path.exists():
        with open(path, 'rb') as f:
            data = pickle.load(f)
        if isinstance(data, dict):
            return data
    return {}

def save_all_training_histories(path, all_histories):
    with open(path, 'wb') as f:
        pickle.dump(all_histories, f)

def load_completion_status(path):
    default_status = {'completed_ratios': [], 'completed_years': {}}
    if path.exists():
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if isinstance(data, dict):
            if 'completed_ratios' not in data:
                data['completed_ratios'] = []
            if 'completed_years' not in data:
                data['completed_years'] = {}
            return data
    return default_status

def save_completion_status(path, status):
    status['completed_ratios'] = sorted(list(set(status.get('completed_ratios', []))))
    cleaned_completed_years = {}
    for key, years in status.get('completed_years', {}).items():
        cleaned_completed_years[key] = sorted(list(set(int(y) for y in years)))
    status['completed_years'] = cleaned_completed_years

    with open(path, 'w', encoding='utf-8') as f:
        json.dump(status, f, indent=2)

def append_training_log(path, message):
    timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(path, 'a', encoding='utf-8') as f:
        f.write(f'[{timestamp}] {message}\n')

def format_ratio_key(replay_ratio):
    return f'RR_{replay_ratio:.1f}'

def build_mlp_model():
    return MLPClassifier(
        hidden_layer_sizes=(64,),
        activation='relu',
        alpha=0.0001,
        random_state=42,
        solver='adam',
        learning_rate='adaptive',
        max_iter=1,
        learning_rate_init=0.001,
        warm_start=False,
        verbose=False,
    )

def get_cached_raw_year(cache, year_idx):
    if year_idx not in cache:
        return np.empty((0, 0), dtype=np.float32), np.empty((0,), dtype=np.int64)
    return cache[year_idx]

def compute_optimal_f1_threshold(y_true, y_proba, threshold_grid, default_threshold=0.5):
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return float(default_threshold)

    from sklearn.metrics import precision_recall_curve
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = np.argmax(f1_scores)
    if best_idx < len(thresholds):
        best_threshold = float(thresholds[best_idx])
    else:
        best_threshold = float(default_threshold)
    return best_threshold

def weighted_choice_without_replacement(indices, weights, sample_size, rng):
    if sample_size <= 0 or len(indices) == 0:
        return np.empty((0,), dtype=np.int64)

    sample_size = min(int(sample_size), len(indices))
    weights = np.asarray(weights, dtype=np.float64)
    weights = np.where(np.isnan(weights) | (weights < 0), 0.0, weights)
    total_weight = float(weights.sum())

    if total_weight <= 0:
        selected_positions = rng.choice(len(indices), size=sample_size, replace=False)
    else:
        non_zero_mask = weights > 0
        n_non_zero = np.sum(non_zero_mask)
        if n_non_zero < sample_size:
            chosen_non_zero = np.where(non_zero_mask)[0]
            remaining_needed = sample_size - n_non_zero
            zero_indices = np.where(~non_zero_mask)[0]
            chosen_zero = rng.choice(zero_indices, size=remaining_needed, replace=False)
            selected_positions = np.concatenate([chosen_non_zero, chosen_zero])
            rng.shuffle(selected_positions)
        else:
            prob = weights / total_weight
            selected_positions = rng.choice(len(indices), size=sample_size, replace=False, p=prob)

    return np.asarray(indices, dtype=np.int64)[selected_positions]

def sample_uncertain_replay_indices(
    model,
    X_replay_pool,
    y_replay_pool,
    replay_year_spans,
    uncertain_target_size,
    rng,
    gaussian_std,
    threshold_grid,
):
    if uncertain_target_size <= 0 or len(replay_year_spans) == 0:
        return np.empty((0,), dtype=np.int64), []

    n_years_with_data = len(replay_year_spans)
    base_quota = uncertain_target_size // n_years_with_data
    remainder = uncertain_target_size % n_years_with_data

    per_year_targets = []
    for idx, year_span in enumerate(replay_year_spans):
        year_idx, start, end = year_span
        year_target = base_quota + (1 if idx < remainder else 0)
        year_size = end - start
        per_year_targets.append((year_idx, start, end, min(year_target, year_size)))

    selected_indices = []
    threshold_values = []

    for year_idx, start, end, year_target in per_year_targets:
        if year_target <= 0:
            continue

        year_indices = np.arange(start, end, dtype=np.int64)
        y_year = y_replay_pool[start:end]
        y_proba_year = model.predict_proba(X_replay_pool[start:end])[:, 1]
        threshold = compute_optimal_f1_threshold(
            y_true=y_year,
            y_proba=y_proba_year,
            threshold_grid=threshold_grid,
            default_threshold=0.5,
        )
        threshold_values.append(threshold)

        distances = np.abs(y_proba_year - threshold)
        if len(distances) > 0 and year_target > 0:
            k_idx = min(int(year_target) - 1, len(distances) - 1)
            d_k = np.partition(distances, k_idx)[k_idx]
            std = max(float(gaussian_std), d_k / 4.8)
        else:
            std = float(gaussian_std)

        gaussian_weights = np.exp(-(distances ** 2) / (2.0 * (std ** 2) + EPSILON))
        chosen = weighted_choice_without_replacement(
            indices=year_indices,
            weights=gaussian_weights,
            sample_size=year_target,
            rng=rng,
        )
        if len(chosen) > 0:
            selected_indices.append(chosen)

    if selected_indices:
        uncertain_indices = np.concatenate(selected_indices).astype(np.int64, copy=False)
    else:
        uncertain_indices = np.empty((0,), dtype=np.int64)

    shortfall = int(uncertain_target_size) - len(uncertain_indices)
    if shortfall > 0:
        all_indices = np.arange(len(y_replay_pool), dtype=np.int64)
        remaining_indices = np.setdiff1d(all_indices, uncertain_indices, assume_unique=False)
        if len(remaining_indices) > 0:
            top_up = rng.choice(remaining_indices, size=min(shortfall, len(remaining_indices)), replace=False)
            uncertain_indices = np.concatenate([uncertain_indices, top_up.astype(np.int64, copy=False)])

    return uncertain_indices, threshold_values

def sample_random_replay_indices(replay_pool_size, random_target_size, excluded_indices, rng):
    if random_target_size <= 0 or replay_pool_size <= 0:
        return np.empty((0,), dtype=np.int64)

    all_indices = np.arange(replay_pool_size, dtype=np.int64)
    available_indices = np.setdiff1d(all_indices, excluded_indices, assume_unique=False)
    if len(available_indices) == 0:
        return np.empty((0,), dtype=np.int64)

    take = min(int(random_target_size), len(available_indices))
    return rng.choice(available_indices, size=take, replace=False).astype(np.int64, copy=False)

all_training_histories = load_all_training_histories(ALL_HISTORIES_FILE)
completion_status = load_completion_status(COMPLETION_STATUS_FILE)

n_years = len(ds.year)
year_values = ds.year.values
year_value_to_idx = {int(y): idx for idx, y in enumerate(year_values)}
all_target_year_values = [int(year_values[idx]) for idx in range(1, n_years)]

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Replay ratios: {REPLAY_RATIOS}')
append_training_log(TRAINING_LOG_FILE, f'Started training run for ratios: {REPLAY_RATIOS}')

for ratio_idx, replay_ratio in enumerate(REPLAY_RATIOS):
    ratio_key = format_ratio_key(replay_ratio)
    output_suffix = f'incremental_scaler_experience_replay_uncertainity_prioritization_{ratio_key}'
    models_dir_name = f'models_mlp_prevyears_monthly_features_{output_suffix}'
    scaler_file_template = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    model_file_template = f'model_year_{{year}}_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_scaler_filename = f'scaler_final_mlp_prevyears_monthly_features_{output_suffix}.pkl'
    final_model_filename = f'mlp_classifier_model_prevyears_monthly_features_{output_suffix}.pkl'
    history_filename = f'mlp_classifier_history_prevyears_monthly_features_{output_suffix}.csv'

    models_dir = Path('.') / models_dir_name
    models_dir.mkdir(exist_ok=True)

    if ratio_key in completion_status.get('completed_ratios', []):
        print(f'[{ratio_key}] already completed. Skipping ratio.')
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] skipped (already completed).')
        continue

    training_history = all_training_histories.get(ratio_key, create_empty_training_history())
    for key in create_empty_training_history().keys():
        training_history.setdefault(key, [])

    completed_years = set(int(y) for y in completion_status.get('completed_years', {}).get(ratio_key, []))
    completed_years.update(int(y) for y in training_history.get('year', []))
    completion_status.setdefault('completed_years', {})[ratio_key] = sorted(list(completed_years))

    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + ratio_idx)

    model = None
    incremental_scaler = None
    start_year_idx = 1

    if completed_years:
        resume_candidate_years = sorted(completed_years, reverse=True)
        resumed = False
        for resume_year in resume_candidate_years:
            year_model_path = models_dir / model_file_template.format(year=resume_year)
            year_scaler_path = models_dir / scaler_file_template.format(year=resume_year)
            if year_model_path.exists() and year_scaler_path.exists() and resume_year in year_value_to_idx:
                with open(year_model_path, 'rb') as f:
                    model = pickle.load(f)
                with open(year_scaler_path, 'rb') as f:
                    incremental_scaler = pickle.load(f)
                start_year_idx = year_value_to_idx[resume_year] + 1
                resumed = True
                print(f'[{ratio_key}] resuming from year {resume_year}; continuing at index {start_year_idx}.')
                append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] resumed from year {resume_year}.')
                break

        if resumed:
            # Clean up completed_years and training_history for years after the resumed year
            completed_years = {y for y in completed_years if y <= resume_year}
            if training_history.get('year'):
                keep_indices = [i for i, y in enumerate(training_history['year']) if y <= resume_year]
                for key in training_history.keys():
                    if isinstance(training_history[key], list):
                        training_history[key] = [training_history[key][i] for i in keep_indices]
        else:
            print(f'[{ratio_key}] checkpoint artifacts missing/inconsistent. Restarting this ratio from scratch.')
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] restart due to missing/inconsistent checkpoint artifacts.')
            completed_years = set()
            completion_status['completed_years'][ratio_key] = []
            training_history = create_empty_training_history()

    if model is None:
        model = build_mlp_model()
    if incremental_scaler is None:
        incremental_scaler = StandardScaler()

    print(f'[{ratio_key}] Model output directory: {models_dir.resolve()}')
    print(f'[{ratio_key}] Years to train: 1..{n_years - 1} (year 0 skipped)')

    for year_idx in tqdm(range(start_year_idx, n_years), desc=f'{ratio_key} by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years:
            continue

        X_train_raw, y_train_batch = get_cached_raw_year(train_feature_cache, year_idx)
        X_val_raw, y_val_batch = get_cached_raw_year(val_feature_cache, year_idx)

        if len(X_train_raw) == 0 or len(X_val_raw) == 0:
            print(f'[{ratio_key}] Year {year_val}: skipped (empty after filtering)')
            completed_years.add(year_val)
            completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
            save_completion_status(COMPLETION_STATUS_FILE, completion_status)
            append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] year {year_val} skipped (empty after filtering).')
            continue

        incremental_scaler.partial_fit(X_train_raw)
        X_train_batch = incremental_scaler.transform(X_train_raw)
        X_val_batch = incremental_scaler.transform(X_val_raw)

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_scaler_path = models_dir / scaler_file_template.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        replay_X_parts = []
        replay_y_parts = []
        replay_year_spans = []
        replay_cursor = 0

        if REPLAY_ENABLED and year_idx > 1:
            for past_year_idx in range(1, year_idx):
                X_past_raw, y_past = get_cached_raw_year(train_feature_cache, past_year_idx)
                if len(X_past_raw) > 0:
                    X_past = incremental_scaler.transform(X_past_raw)
                    replay_X_parts.append(X_past)
                    replay_y_parts.append(y_past)
                    span_len = len(y_past)
                    replay_year_spans.append((past_year_idx, replay_cursor, replay_cursor + span_len))
                    replay_cursor += span_len

        if replay_X_parts:
            X_replay_pool = np.vstack(replay_X_parts)
            y_replay_pool = np.concatenate(replay_y_parts)
        else:
            X_replay_pool = np.empty((0, X_train_batch.shape[1]), dtype=X_train_batch.dtype)
            y_replay_pool = np.empty((0,), dtype=y_train_batch.dtype)

        replay_pool_size = len(y_replay_pool)
        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None
        uncertain_used_size = 0
        random_used_size = 0
        uncertain_target_size = 0
        random_target_size = 0
        uncertain_threshold_mean = np.nan

        for epoch in range(MAX_EPOCHS):
            if replay_used_size > 0:
                uncertain_target_size = int(np.floor(replay_used_size * UNCERTAIN_REPLAY_FRACTION))
                random_target_size = int(replay_used_size - uncertain_target_size)

                uncertain_indices, threshold_values = sample_uncertain_replay_indices(
                    model=model,
                    X_replay_pool=X_replay_pool,
                    y_replay_pool=y_replay_pool,
                    replay_year_spans=replay_year_spans,
                    uncertain_target_size=uncertain_target_size,
                    rng=replay_rng,
                    gaussian_std=UNCERTAINTY_GAUSSIAN_STD,
                    threshold_grid=UNCERTAINTY_THRESHOLD_GRID,
                )
                random_indices = sample_random_replay_indices(
                    replay_pool_size=replay_pool_size,
                    random_target_size=random_target_size,
                    excluded_indices=uncertain_indices,
                    rng=replay_rng,
                )

                replay_indices = np.concatenate([uncertain_indices, random_indices])
                if len(replay_indices) > 0:
                    replay_indices = replay_indices.astype(np.int64, copy=False)
                    replay_indices = replay_indices[replay_rng.permutation(len(replay_indices))]

                uncertain_used_size = int(len(uncertain_indices))
                random_used_size = int(len(random_indices))
                if threshold_values:
                    uncertain_threshold_mean = float(np.mean(threshold_values))
                else:
                    uncertain_threshold_mean = np.nan

                if len(replay_indices) > 0:
                    X_replay_sampled = X_replay_pool[replay_indices]
                    y_replay_sampled = y_replay_pool[replay_indices]
                    X_combined = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
                    y_combined = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
                else:
                    X_combined = X_train_batch
                    y_combined = y_train_batch
            else:
                X_combined = X_train_batch
                y_combined = y_train_batch

            combined_n_samples = len(X_combined)
            shuffle_idx = replay_rng.permutation(combined_n_samples)
            X_train_shuffled = X_combined[shuffle_idx]
            y_train_shuffled = y_combined[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                sample_weights_chunk = np.array([class_weight_dict[int(label)] for label in y_chunk])
                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            y_val_pred = model.predict(X_val_batch)
            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = {
                    'coefs': [w.copy() for w in model.coefs_],
                    'intercepts': [b.copy() for b in model.intercepts_],
                    'n_layers_': model.n_layers_,
                    'n_outputs_': getattr(model, 'n_outputs_', None),
                    'out_activation_': getattr(model, 'out_activation_', None),
                }
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    if best_model_state is not None:
                        model.coefs_ = [w.copy() for w in best_model_state['coefs']]
                        model.intercepts_ = [b.copy() for b in best_model_state['intercepts']]
                        model.n_layers_ = best_model_state['n_layers_']
                        if best_model_state['n_outputs_'] is not None:
                            model.n_outputs_ = best_model_state['n_outputs_']
                        if best_model_state['out_activation_'] is not None:
                            model.out_activation_ = best_model_state['out_activation_']
                    break

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))
        training_history['uncertain_target_size'].append(int(uncertain_target_size))
        training_history['uncertain_used_size'].append(int(uncertain_used_size))
        training_history['random_target_size'].append(int(random_target_size))
        training_history['random_used_size'].append(int(random_used_size))
        training_history['uncertain_threshold_mean'].append(float(uncertain_threshold_mean) if not np.isnan(uncertain_threshold_mean) else np.nan)

        year_model_path = models_dir / model_file_template.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        completed_years.add(year_val)
        completion_status['completed_years'][ratio_key] = sorted(list(completed_years))
        all_training_histories[ratio_key] = training_history.copy()
        save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)

        print(
            f'[{ratio_key}] Year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, Val PR-AUC={val_pr_auc:.3f}, '
            f'replay_used={replay_used_size:,}/{replay_pool_size:,}, uncertain={uncertain_used_size:,}, random={random_used_size:,}'
        )
        append_training_log(
            TRAINING_LOG_FILE,
            f'[{ratio_key}] completed year {year_val} with replay_used={replay_used_size}/{replay_pool_size}, '
            f'uncertain={uncertain_used_size}, random={random_used_size}, gaussian_std={UNCERTAINTY_GAUSSIAN_STD}.',
        )

    if hasattr(incremental_scaler, 'mean_'):
        final_scaler_path = models_dir / final_scaler_filename
        with open(final_scaler_path, 'wb') as f:
            pickle.dump(incremental_scaler, f)
        print(f'[{ratio_key}] Final incremental scaler saved: {final_scaler_path.name}')

    final_model_path = models_dir / final_model_filename
    with open(final_model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'[{ratio_key}] Final model saved: {final_model_path.name}')

    ratio_history_df = pd.DataFrame(training_history).sort_values('year').reset_index(drop=True)
    ratio_history_path = models_dir / history_filename
    ratio_history_df.to_csv(ratio_history_path, index=False)
    print(f'[{ratio_key}] History saved: {ratio_history_path}')

    if set(all_target_year_values).issubset(completed_years):
        if ratio_key not in completion_status.get('completed_ratios', []):
            completion_status.setdefault('completed_ratios', []).append(ratio_key)
        save_completion_status(COMPLETION_STATUS_FILE, completion_status)
        append_training_log(TRAINING_LOG_FILE, f'[{ratio_key}] marked as fully completed.')

    all_training_histories[ratio_key] = training_history.copy()
    save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)

combined_history_frames = []
for ratio_key, history_dict in all_training_histories.items():
    if not isinstance(history_dict, dict):
        continue
    if len(history_dict.get('year', [])) == 0:
        continue
    df_ratio = pd.DataFrame(history_dict).sort_values('year').reset_index(drop=True)
    replay_ratio = float(ratio_key.split('_')[1])
    df_ratio['ratio_key'] = ratio_key
    df_ratio['replay_ratio'] = replay_ratio
    combined_history_frames.append(df_ratio)

if combined_history_frames:
    combined_history_df = pd.concat(combined_history_frames, ignore_index=True)
    combined_history_df = combined_history_df.sort_values(['replay_ratio', 'year']).reset_index(drop=True)
    combined_history_path = Path('.') / COMBINED_HISTORY_FILENAME
    combined_history_df.to_csv(combined_history_path, index=False)
    print(f'Combined history saved: {combined_history_path}')
    print(f'Combined rows: {len(combined_history_df):,}')
else:
    print('No history data available to write combined history.')

save_completion_status(COMPLETION_STATUS_FILE, completion_status)
save_all_training_histories(ALL_HISTORIES_FILE, all_training_histories)
append_training_log(TRAINING_LOG_FILE, 'Training run completed.')

Checkpoint directory: C:\Users\bartu\Desktop\Fonda-scikit - Git\training_checkpoints_mlp_experience_replay_uncertainity_prioritization
Replay ratios: [0.2, 0.3, 0.4, 0.5]
[RR_0.2] already completed. Skipping ratio.
[RR_0.3] already completed. Skipping ratio.
[RR_0.4] already completed. Skipping ratio.
[RR_0.5] already completed. Skipping ratio.


Combined history saved: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_uncertainity_prioritization_all_ratios.csv
Combined rows: 1


## 5. Save Training History

In [6]:
import json

CHECKPOINT_DIR = Path('.') / 'training_checkpoints_mlp_experience_replay_uncertainity_prioritization'
ALL_HISTORIES_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories_uncertainity_prioritization.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status_uncertainity_prioritization.json'
COMBINED_HISTORY_FILENAME = 'mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_uncertainity_prioritization_all_ratios.csv'

if ALL_HISTORIES_FILE.exists():
    with open(ALL_HISTORIES_FILE, 'rb') as f:
        all_training_histories = pickle.load(f)
else:
    all_training_histories = {}

if COMPLETION_STATUS_FILE.exists():
    with open(COMPLETION_STATUS_FILE, 'r', encoding='utf-8') as f:
        completion_status = json.load(f)
else:
    completion_status = {'completed_ratios': [], 'completed_years': {}}

summary_rows = []
for ratio_key in sorted(all_training_histories.keys()):
    history_dict = all_training_histories[ratio_key]
    n_rows = len(history_dict.get('year', []))
    last_year = history_dict['year'][-1] if n_rows > 0 else np.nan
    last_val_f1 = history_dict['val_f1'][-1] if n_rows > 0 else np.nan
    last_val_pr_auc = history_dict['val_pr_auc'][-1] if n_rows > 0 else np.nan
    is_completed = ratio_key in completion_status.get('completed_ratios', [])
    summary_rows.append(
        {
            'ratio_key': ratio_key,
            'rows': n_rows,
            'last_year': last_year,
            'last_val_f1': last_val_f1,
            'last_val_pr_auc': last_val_pr_auc,
            'completed': is_completed,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values('ratio_key').reset_index(drop=True)
print('Per-ratio training summary:')
display(summary_df)

combined_history_path = Path('.') / COMBINED_HISTORY_FILENAME
if combined_history_path.exists():
    combined_df = pd.read_csv(combined_history_path)
    print(f'Combined history file: {combined_history_path}')
    print(f'Rows: {len(combined_df):,}')
    display(combined_df.tail())
else:
    print(f'Combined history file not found: {combined_history_path}')

Per-ratio training summary:


,ratio_key,rows,last_year,last_val_f1,last_val_pr_auc,completed
0,RR_0.2,1,2017.0,0.135038,0.180609,True
1,RR_0.3,0,NaN,NaN,NaN,True
2,RR_0.4,0,NaN,NaN,NaN,True
3,RR_0.5,0,NaN,NaN,NaN,True


Combined history file: mlp_classifier_history_prevyears_monthly_features_incremental_scaler_experience_replay_uncertainity_prioritization_all_ratios.csv
Rows: 1


,year,train_accuracy,train_precision,train_recall,train_f1,val_accuracy,val_precision,val_recall,val_f1,val_roc_auc,...,year_positive_rate,replay_actual_positive_rate,replay_pool_positive_rate,current_class_weight_0,current_class_weight_1,replay_class_weight_0,replay_class_weight_1,replay_weight_fallback,ratio_key,replay_ratio
0,2017,0.836185,0.081374,0.76509,0.147102,0.843761,0.075369,0.648277,0.135038,0.832138,...,0.018464,NaN,NaN,0.509406,27.079398,NaN,NaN,no_replay,RR_0.2,0.2
